# Jarvis — treinamento LoRA na nuvem

Notebook pensado para executar pelo navegador do celular usando Google Colab. Usa uma GPU de nuvem quando ela estiver disponível. **A disponibilidade de GPU no Colab gratuito não é garantida.**

Este processo ajusta um adaptador LoRA de `Qwen/Qwen3-0.6B`; não treina um modelo do zero. Os arquivos finais são salvos no Google Drive. Não coloque tokens, senhas ou dados pessoais no notebook.


## 1. Conectar o Google Drive
Execute a célula e autorize o acesso para guardar o resultado mesmo se a sessão do Colab acabar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUTPUT_DIR = Path('/content/drive/MyDrive/Jarvis-training/jarvis-intensive-lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Destino do adaptador:', OUTPUT_DIR)


## 2. Baixar o repositório e verificar o ambiente

In [ ]:
import subprocess, sys, os
REPO_DIR = Path('/content/Jarvis-mobile')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/viole777/Jarvis-mobile.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'GPU não detectada; abra Ambiente de execução > Alterar tipo de ambiente de execução e selecione GPU, se disponível.')


## 3. Instalar dependências compatíveis
Esta célula evita reinstalar o PyTorch pré-configurado pelo Colab. Se o treino falhar por incompatibilidade, guarde a mensagem de erro antes de alterar versões.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'datasets==3.6.0', 'trl==0.18.2', 'peft==0.15.2', 'accelerate==1.6.0', 'tokenizers>=0.21.0'], check=True)
print('Dependências instaladas. Se necessário, reinicie o ambiente e execute as células novamente.')


## 4. Validar os dados antes de treinar
A base de treino tem cerca de 199 exemplos e a validação apenas 12. Isso é suficiente para um experimento inicial, não para provar capacidade geral.

In [ ]:
TRAIN = REPO_DIR / 'ml/data/seed_combined_train.jsonl'
VALIDATION = REPO_DIR / 'ml/data/seed_validation.jsonl'
subprocess.run([sys.executable, 'ml/validate_dataset.py', str(TRAIN)], check=True)
subprocess.run([sys.executable, 'ml/validate_dataset.py', str(VALIDATION)], check=True)


## 5. Executar o ajuste LoRA
Configuração inicial: 5 épocas, batch 1, acumulação de gradiente 8 e taxa de aprendizado `5e-5`. Pode demorar; mantenha o Colab aberto e não interrompa a célula. Se a GPU não estiver disponível, não inicie o treino pesado em CPU.

In [ ]:
TRAIN_SCRIPT = REPO_DIR / 'ml/train_lora.py'
cmd = [sys.executable, str(TRAIN_SCRIPT), '--train', str(TRAIN), '--validation', str(VALIDATION), '--output', str(OUTPUT_DIR), '--epochs', '5', '--batch-size', '1', '--gradient-accumulation', '8', '--learning-rate', '5e-5']
print('Iniciando treinamento. Saída será salva no Drive:', OUTPUT_DIR)
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Treinamento terminou sem erro de processo. Ainda é necessário avaliar as métricas.')


## 6. Avaliar o adaptador salvo
Esta validação é pequena e não deve ser confundida com uma avaliação independente robusta. Compare resultados com o modelo-base antes de concluir que o ajuste melhorou o Jarvis.

In [ ]:
PREDICTIONS = OUTPUT_DIR / 'seed-validation-predictions.jsonl'
subprocess.run([sys.executable, str(REPO_DIR / 'ml/evaluate.py'), '--dataset', str(VALIDATION), '--model', str(OUTPUT_DIR), '--predictions', str(PREDICTIONS)], cwd=REPO_DIR, check=True)
print('Predições:', PREDICTIONS)
print('Adaptador e checkpoints devem estar em:', OUTPUT_DIR)


## Limitações e próximo passo
- Este notebook não integra automaticamente o adaptador ao app Android ou à versão web.
- O treino ajusta um modelo pré-treinado; não cria uma IA geral do zero.
- Os exemplos são poucos e precisam ser ampliados e revisados. Não publique métricas sem registrar a configuração e o resultado.
- Guarde o adaptador no Drive e depois transfira os arquivos para um armazenamento seguro ou release do projeto; não faça commit de checkpoints grandes no GitHub.
- Se aparecer erro, copie a mensagem completa do traceback e envie para análise antes de tentar várias mudanças aleatórias.
